In [1]:
import os
import torch
import mlflow
import dagshub
from ultralytics import YOLO
from ultralytics import settings

In [2]:
torch.cuda.empty_cache()

In [3]:
MY_USERNAME = "TQanh23" 
REPO_NAME = "ppe_v2"

In [ ]:
dagshub.init(repo_owner=MY_USERNAME, repo_name=REPO_NAME, mlflow=True)
settings.update({'mlflow': True})
mlflow.set_experiment("PPE__yolov8m_Baseline_Evaluation")

Accessing as TQanh23

Initialized MLflow to track repo "TQanh23/ppe_v2"

Repository TQanh23/ppe_v2 initialized!

In [ ]:
with mlflow.start_run(run_name="YOLOv8m_150e_Baseline_Val") as run:
    model_path = 'downloads/best.pt'
    model = YOLO(model_path)
    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)
    try:
        _, params, _, flops = model.info(verbose=False)
    except Exception:
        params = sum(p.numel() for p in model.model.parameters())
        flops = 0.0
    print("Đang tiến hành đánh giá và log dữ liệu lên DagsHub...")
    metrics = model.val(
        data='data.yaml',
        device=0,       
        batch=4,
        imgsz=640,
        split='val',
        project='PPE_Detection',
        name='baseline_evaluation'
    )

2026/07/25 14:36:48 INFO mlflow.tracking.fluent: Experiment with name 'PPE__yolov8m_Baseline_Evaluation' does not exist. Creating a new experiment.


Đang tiến hành đánh giá và log dữ liệu lên DagsHub...
Ultralytics 8.4.104  Python-3.12.7 torch-2.13.0+cu126 CUDA:0 (NVIDIA GeForce RTX 3050 Ti Laptop GPU, 4096MiB)
Model summary (fused): 93 layers, 25,844,971 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 377.0258.0 MB/s, size: 77.2 KB)
val: Scanning D:\ppe_cv\train_data\valid\labels.cache... 229 images, 19 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 229/229  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 58/58 7.2it/s 8.0s0.1s
                   all        229       2019      0.743      0.689      0.716      0.357
                 boots         93        201      0.584      0.567      0.549      0.231
                gloves         78        157      0.639      0.451      0.481      0.181
                helmet        130        337      0.828      0.828      0.861      0.504
              no_boots         32         65      0.757    

In [ ]:
map50 = metrics.box.map50
map50_95 = metrics.box.map
precision = metrics.box.mp
recall = metrics.box.mr
f1_score = 2 * (precision * recall) / (precision + recall + 1e-16)
inference_time = metrics.speed['inference']

In [ ]:
mlflow.log_params({
        "model_type": "YOLOv8m",
        "imgsz": 640,
        "batch_size": 4,
        "parameters": params,
        "FLOPs_G": flops,
        "model_size_MB": round(model_size_mb, 2)
    })
mlflow.log_metrics({
    "mAP_50": map50,
    "mAP_50_95": map50_95,
    "Precision": precision,
    "Recall": recall,
    "F1_score": f1_score,
    "Inference_Time_ms": inference_time
})
print("Đang upload file trọng số và đồ thị lên DagsHub...")
mlflow.log_artifact(model_path, artifact_path="models")
mlflow.log_artifacts(metrics.save_dir, artifact_path="validation_graphs")

In [ ]:
print("\n" + "="*50)
print(" BÁO CÁO KẾT QUẢ BASELINE TỔNG HỢP ".center(50))
print("="*50)
print(f"Run ID:                  {run.info.run_id}")
print("-" * 50)
print(f"mAP@50:                  {map50:.4f}")
print(f"mAP@50-95:               {map50_95:.4f}")
print(f"Precision (P):           {precision:.4f}")
print(f"Recall (R):              {recall:.4f}")
print(f"F1-score:                {f1_score:.4f}")
print("-" * 50)
print(f"Kích thước mô hình:      {model_size_mb:.2f} MB")
print(f"Số lượng tham số:        {params:,.0f}")
print(f"FLOPs:                   {flops:.1f} GFLOPs")
print(f"Thời gian suy luận:      {inference_time:.2f} ms/ảnh")
print("="*50)